In [3]:
import awswrangler as wr

In [2]:
def exec_sql(sql_statement:str, db="portfolio-dev"):
  return wr.athena.read_sql_query(
        sql=sql_statement,
        database=db
    )

In [5]:
sql ="""
    SELECT *
    FROM silver_t212_positions
    ORDER BY ingested_date DESC
    limit 10;
"""

df_positions = exec_sql(sql)

In [10]:
sql ="""
    SELECT *
    FROM silver_t212_account_summary
    ORDER BY ingested_date DESC
    limit 10;
"""

df_account = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


In [9]:
df_account.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   id                        10 non-null     Int64         
 1   currency                  10 non-null     string        
 2   total_value               10 non-null     float64       
 3   cash_available_to_trade   10 non-null     float64       
 4   cash_reserved_for_orders  10 non-null     Int64         
 5   cash_in_pies              10 non-null     float64       
 6   total_value_investmented  10 non-null     float64       
 7   total_investment_cost     10 non-null     float64       
 8   realized_profit_loss      10 non-null     float64       
 9   unrealized_profit_loss    10 non-null     float64       
 10  ingested_timestamp        10 non-null     datetime64[ns]
 11  ingested_date             10 non-null     object        
dtypes: Int64(2), datetime64[ns](1), floa

In [11]:
sql ="""
    WITH account_summary AS (
      SELECT 
          ROW_NUMBER() OVER (PARTITION BY ingested_date ORDER BY ingested_timestamp DESC) as rn
        , total_value_investmented
        , ingested_date
      FROM silver_t212_account_summary
    limit 10
    )
    SELECT *
    FROM account_summary
    WHERE rn = 1
    ORDER BY ingested_date DESC
"""

df_account_summary = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


In [97]:
df_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   name                            10 non-null     string        
 1   ticker                          10 non-null     string        
 2   isin                            10 non-null     string        
 3   created_at                      10 non-null     datetime64[ns]
 4   asset_currency                  10 non-null     string        
 5   avg_price_paid                  10 non-null     float64       
 6   current_price                   10 non-null     float64       
 7   quantity                        10 non-null     float64       
 8   quantity_available_for_trading  10 non-null     float64       
 9   quantity_in_pies                10 non-null     float64       
 10  account_currency                10 non-null     string        
 11  current_value       

In [4]:
from datetime import datetime, timezone

current_date = datetime.now(timezone.utc).date()

sql =F"""
SELECT name
, ticker
, weight_pct
, daily_value_change_pct
, current_value
, avg_value_7d
, pnl
,ingested_timestamp
FROM fact_t212_positions
WHERE
    ingested_date = date_parse('{current_date}', '%Y-%m-%d')
  AND daily_value_change_pct <= -1
  AND current_value < avg_value_7d 
ORDER BY ingested_timestamp, daily_value_change_pct, weight_pct
"""

df_fact = exec_sql(sql)

df_fact.head(3)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,name,ticker,weight_pct,daily_value_change_pct,current_value,avg_value_7d,pnl,ingested_timestamp
0,Adobe,ADBE_US_EQ,0.185725,-3.014184,16.41,17.115714,-4.35,2026-10-27 19:32:50.590
1,Sony,SNE_US_EQ,0.213227,-2.180685,18.84,18.917143,1.75,2026-10-27 19:32:50.590
2,Chevron,CHVd_EQ,0.918438,-1.564774,81.15,81.521429,22.15,2026-10-27 19:32:50.590


In [307]:
# x target allocation is 6
# equity 100

equity = 120
target_weight = 6.00
current_weight = 5.287571	

df["top_up"] = equity * (abs(target_weight - df["weight_pct"])) / 100

df

,name,ticker,weight_pct,daily_value_change_pct,current_value,avg_value_7d,pnl,ingested_timestamp,top_up,target_value,rebalance_amount
4,Chevron,CHVd_EQ,0.863769,-1.404572,1.036523,72.037143,12.60,2026-09-25 14:35:45.129,6.163477,6.0,4.963477
11,Invesco Energy S&P US Select Sector (Acc),XLEPl_EQ,5.287571,-2.296032,6.345086,445.702857,42.73,2026-09-25 14:35:45.129,0.854914,6.0,-0.345086
19,British American Tobacco,BMT1d_EQ,0.606569,-1.527615,0.727882,50.594286,0.10,2026-09-25 14:35:45.129,6.472118,6.0,5.272118


In [308]:
# Calculate dollar values
df["target_value"] = equity * (target_weight / 100)
df["current_value"] = equity * (df["weight_pct"] / 100)
# Positive = Need to BUY, Negative = Need to SELL
df["rebalance_amount"] = df["target_value"] - df["current_value"]
print(df[["ticker", "current_value", "weight_pct", "target_value", "rebalance_amount"]])

      ticker  current_value  weight_pct  target_value  rebalance_amount
4    CHVd_EQ       1.036523    0.863769           7.2          6.163477
11  XLEPl_EQ       6.345086    5.287571           7.2          0.854914
19  BMT1d_EQ       0.727882    0.606569           7.2          6.472118


In [269]:
(target_weight - current_weight) / 100


0.007124290000000002

In [ ]:
from_date = "2026-08-17"
to_date = "2026-09-25"

sql =F"""
  SELECT t1.*, t2.
  FROM dim_t212_asset t1
    INNER JOIN fact_t212_positions t2
      ON t1.ticker = t2.ticker
  LIMIT 10;
  
"""

t212_asset_dim = exec_sql(sql)

/Users/kunmi/workspace/projects/engineering/repository/portfolio-platform/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


InvalidCtasApproachQuery: Please, define distinct names for your columns. Root error message: DUPLICATE_COLUMN_NAME: line 1:1: Column name 'name' specified more than once. You may need to manually clean the data at location 's3://aws-athena-query-results-861580917950-eu-west-1/tables/421ccaec-886a-4f56-9435-d245b9a75c73' before retrying. Athena will not delete data in your account.

In [14]:
t212_asset_dim

,ticker,symbol,name,sector,industry,asset_type,is_etf
0,AMDd_EQ,AMD,Advanced Micro Devices,Information Technology,Semiconductors,stock,False
1,CBUKd_EQ,CBUK,iShares MSCI China Tech (Acc),Information Technology,China Technology,etf,True
2,UNH_US_EQ,UNH,UnitedHealth,Healthcare,Managed Healthcare,stock,False
3,IH2Ol_EQ,IH2O,iShares Global Water (Dist),Utilities,Water,etf,True
4,MSFT_US_EQ,MSFT,Microsoft,Information Technology,Software,stock,False
5,VUSAl_EQ,VUSA,Vanguard S&P 500 (Dist),Multi-sector,Broad Market Equity,etf,True
6,WQTMd_EQ,WQTM,WisdomTree Quantum Computing (Acc),Information Technology,Quantum Computing,etf,True
7,GOOGL_US_EQ,GOOGL,Alphabet Class A,Communication Services,Internet Services,stock,False
8,NKE_US_EQ,NKE,Nike,Consumer Discretionary,Apparel & Footwear,stock,False
9,ISLNl_EQ,ISLN,iShares Physical Silver,Commodities,Precious Metals,etf,True


In [8]:
x = f"""SELECT
  ticker
, weight_pct
, daily_value_change_pct
, current_value
, avg_value_7d
, pnl
,ingested_timestamp
FROM fact_t212_positions
"""

x = exec(x)

IndentationError: unexpected indent (<string>, line 2)